# Planar Graph Permutation + Balanced Augmentation

This Colab notebook performs **two stages**.

### Stage 1 — Find successful permutations
- Reads the original planar-graph TXT file.
- Uses **only the first planar graph (L1)** before `;`.
- For every trial, it **randomly selects 10 nodes again** and creates a fresh permutation.
- The permutation is accepted only when `χ(L1 ∪ permutation(L1))` is **8 or 9**.
- Saves up to **10 successful permutations per input graph**.
- Keeps the adjacency matrices for L1, permuted L1, and their union.
- All displayed node labels are **1-indexed**.

### Stage 2 — Balanced augmentation
- Takes the successful Stage-1 CSV.
- Randomly relabels the vertices of **both L1 and its successful permutation using the same relabeling**.
- Does not change graph structure or chromatic numbers.
- Produces approximately **5,000 augmented samples**.
- If there are exactly 400 base samples, augmentation is balanced as **12 or 13 copies per base sample** (200 each).
- If the actual number is 390, the same balanced rule is applied automatically.


In [1]:
# 1. Install Boost Graph Library
!apt-get update -qq
!apt-get install -y -qq libboost-graph-dev

print("Boost installation complete.")


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libboost-graph1.83.0:amd64.
(Reading database ... 122797 files and directories currently installed.)
Preparing to unpack .../0-libboost-graph1.83.0_1.83.0-2.1ubuntu3.2_amd64.deb ...
Unpacking libboost-graph1.83.0:amd64 (1.83.0-2.1ubuntu3.2) ...
Selecting previously unselected package libboost-regex1.83.0:amd64.
Preparing to unpack .../1-libboost-regex1.83.0_1.83.0-2.1ubuntu3.2_amd64.deb ...
Unpacking libboost-regex1.83.0:amd64 (1.83.0-2.1ubuntu3.2) ...
Selecting previously unselected package libboost-regex1.83-dev:amd64.
Preparing to unpack .../2-libboost-regex1.83-dev_1.83.0-2.1ubuntu3.2_amd64.deb ...
Unpacking libboost-regex1.83-dev:amd64 (1.83.0-2.1ubuntu3.2) ...
Selecting previously unselected package libboost-serialization1.83.0:amd64.
Preparing to unpack 

In [2]:
# 2. Upload the original TXT input file
from google.colab import files
import os

uploaded = files.upload()
INPUT_FILE = next(iter(uploaded.keys()))

STAGE1_CSV = "planar_permutation_results_final.csv"
AUGMENTED_CSV = "augmented_5000_planar_permutation_results.csv"

print("Input file:", INPUT_FILE)
print("Stage-1 output:", STAGE1_CSV)
print("Augmentation output:", AUGMENTED_CSV)


Saving graph level1,2 changed crct values.txt to graph level1,2 changed crct values.txt
Input file: graph level1,2 changed crct values.txt
Stage-1 output: planar_permutation_results_final.csv
Augmentation output: augmented_5000_planar_permutation_results.csv


In [3]:
CPP_FILE = 'planar_permutation_boost_final.cpp'

cpp_source = r'''
#include <algorithm>
#include <chrono>
#include <fstream>
#include <iostream>
#include <numeric>
#include <random>
#include <regex>
#include <set>
#include <sstream>
#include <string>
#include <vector>

using namespace std;
using Clock = chrono::steady_clock;

struct GraphData {
    string id;
    int n;
    vector<vector<int>> adj;
};

vector<int> extractIntegers(const string& s) {
    vector<int> result;
    regex r(R"(\d+)");
    auto begin = sregex_iterator(s.begin(), s.end(), r);
    auto end = sregex_iterator();
    for (auto it = begin; it != end; ++it)
        result.push_back(stoi(it->str()));
    return result;
}

/* Read ONLY L1: the part before ';'. */
vector<GraphData> readGraphs(const string& filename) {
    ifstream fin(filename);
    if (!fin) {
        cerr << "ERROR: Cannot open input file: " << filename << endl;
        exit(1);
    }

    vector<GraphData> graphs;
    string line;
    GraphData current;
    bool reading = false;

    while (getline(fin, line)) {
        if (!line.empty() && line.back() == '\r')
            line.pop_back();

        if (line.rfind("ID", 0) == 0) {
            if (reading)
                graphs.push_back(current);

            current = GraphData();
            current.id = line;

            vector<int> nums = extractIntegers(line);
            current.n = (nums.size() >= 2) ? nums[1] : 0;
            current.adj.assign(current.n, {});
            reading = true;
            continue;
        }

        if (!reading || line.empty())
            continue;

        size_t semicolon = line.find(';');
        string firstLayer =
            (semicolon == string::npos) ? line : line.substr(0, semicolon);

        vector<int> nums = extractIntegers(firstLayer);
        if (nums.empty())
            continue;

        int vertex = nums[0];
        if (vertex < 1 || vertex > current.n)
            continue;

        int u = vertex - 1;
        for (size_t i = 1; i < nums.size(); ++i) {
            int v = nums[i];
            if (v >= 1 && v <= current.n && v != vertex)
                current.adj[u].push_back(v - 1);
        }
    }

    if (reading)
        graphs.push_back(current);

    // Make undirected and remove duplicate edges.
    for (auto& g : graphs) {
        vector<set<int>> sets(g.n);
        for (int u = 0; u < g.n; ++u) {
            for (int v : g.adj[u]) {
                if (v >= 0 && v < g.n && u != v) {
                    sets[u].insert(v);
                    sets[v].insert(u);
                }
            }
        }

        g.adj.assign(g.n, {});
        for (int u = 0; u < g.n; ++u)
            g.adj[u] = vector<int>(sets[u].begin(), sets[u].end());
    }

    return graphs;
}

class ChromaticSolver {
    int n;
    const vector<vector<int>>& adj;
    vector<int> color;
    vector<int> degree;
    Clock::time_point deadline;
    bool timedOut = false;

    int saturation(int v) {
        vector<char> used(n + 1, false);
        for (int u : adj[v])
            if (color[u] >= 0)
                used[color[u]] = true;

        int s = 0;
        for (char x : used) s += x;
        return s;
    }

    int selectVertex() {
        int selected = -1, bestSat = -1, bestDeg = -1;

        for (int v = 0; v < n; ++v) {
            if (color[v] != -1) continue;

            int sat = saturation(v);
            if (sat > bestSat || (sat == bestSat && degree[v] > bestDeg)) {
                selected = v;
                bestSat = sat;
                bestDeg = degree[v];
            }
        }
        return selected;
    }

    bool canUseColor(int v, int c) {
        for (int u : adj[v])
            if (color[u] == c)
                return false;
        return true;
    }

    bool dfsKColor(int k) {
        if (Clock::now() >= deadline) {
            timedOut = true;
            return false;
        }

        int v = selectVertex();
        if (v == -1) return true;

        for (int c = 0; c < k; ++c) {
            if (!canUseColor(v, c)) continue;

            color[v] = c;
            if (dfsKColor(k)) return true;
            color[v] = -1;

            if (timedOut) return false;
        }
        return false;
    }

    int greedyUpperBound() {
        vector<int> order(n);
        iota(order.begin(), order.end(), 0);

        sort(order.begin(), order.end(),
             [&](int a, int b) { return degree[a] > degree[b]; });

        vector<int> gc(n, -1);
        int usedColors = 0;

        for (int v : order) {
            vector<char> used(n + 1, false);
            for (int u : adj[v])
                if (gc[u] >= 0) used[gc[u]] = true;

            int c = 0;
            while (c <= n && used[c]) ++c;
            gc[v] = c;
            usedColors = max(usedColors, c + 1);
        }
        return usedColors;
    }

public:
    ChromaticSolver(const vector<vector<int>>& a, Clock::time_point dl)
        : n((int)a.size()), adj(a), deadline(dl) {
        color.assign(n, -1);
        degree.resize(n);
        for (int i = 0; i < n; ++i)
            degree[i] = (int)adj[i].size();
    }

    int solve() {
        if (n == 0) return 0;

        int upper = greedyUpperBound();
        for (int k = 1; k <= upper; ++k) {
            color.assign(n, -1);
            timedOut = false;

            if (dfsKColor(k))
                return k;

            if (timedOut)
                return -1;
        }
        return upper;
    }
};

int chromaticNumber(const vector<vector<int>>& adj,
                    Clock::time_point deadline) {
    ChromaticSolver solver(adj, deadline);
    return solver.solve();
}

/*
IMPORTANT:
Every call creates a NEW random selection of 10 vertices.
After a successful permutation, the next loop iteration calls this
function again, so the selected nodes are randomly selected again.
*/
vector<int> createPermutation(int n, mt19937& rng) {
    vector<int> selected(n);
    iota(selected.begin(), selected.end(), 0);
    shuffle(selected.begin(), selected.end(), rng);
    selected.resize(10);

    vector<int> shuffled = selected;

    // True derangement among the selected 10 nodes.
    do {
        shuffle(shuffled.begin(), shuffled.end(), rng);
    } while ([&]() {
        for (int i = 0; i < 10; ++i)
            if (shuffled[i] == selected[i])
                return true;
        return false;
    }());

    vector<int> p(n);
    iota(p.begin(), p.end(), 0);

    for (int i = 0; i < 10; ++i)
        p[selected[i]] = shuffled[i];

    return p;
}

vector<vector<int>> permuteGraph(const vector<vector<int>>& adj,
                                 const vector<int>& p) {
    int n = (int)adj.size();
    vector<set<int>> sets(n);

    for (int u = 0; u < n; ++u) {
        for (int v : adj[u]) {
            int nu = p[u];
            int nv = p[v];

            if (nu != nv) {
                sets[nu].insert(nv);
                sets[nv].insert(nu);
            }
        }
    }

    vector<vector<int>> result(n);
    for (int u = 0; u < n; ++u)
        result[u] = vector<int>(sets[u].begin(), sets[u].end());

    return result;
}

vector<vector<int>> unionGraph(const vector<vector<int>>& A,
                               const vector<vector<int>>& B) {
    int n = (int)A.size();
    vector<set<int>> sets(n);

    for (int u = 0; u < n; ++u) {
        for (int v : A[u]) sets[u].insert(v);
        for (int v : B[u]) sets[u].insert(v);
    }

    vector<vector<int>> U(n);
    for (int u = 0; u < n; ++u)
        U[u] = vector<int>(sets[u].begin(), sets[u].end());

    return U;
}

string matrixToCSV(const vector<vector<int>>& adj) {
    int n = (int)adj.size();
    vector<vector<int>> M(n, vector<int>(n, 0));

    for (int u = 0; u < n; ++u)
        for (int v : adj[u]) {
            M[u][v] = 1;
            M[v][u] = 1;
        }

    stringstream ss;
    for (int i = 0; i < n; ++i) {
        if (i) ss << "|";
        for (int j = 0; j < n; ++j) {
            if (j) ss << " ";
            ss << M[i][j];
        }
    }
    return ss.str();
}

/* Human-readable 1-indexed edge list: 1-2, 2-3, ... */
string edgesToCSV(const vector<vector<int>>& adj) {
    vector<pair<int,int>> edges;
    int n = (int)adj.size();

    for (int u = 0; u < n; ++u) {
        for (int v : adj[u]) {
            if (u < v)
                edges.push_back({u + 1, v + 1});
        }
    }

    stringstream ss;
    for (size_t i = 0; i < edges.size(); ++i) {
        if (i) ss << ", ";
        ss << edges[i].first << "-" << edges[i].second;
    }
    return ss.str();
}

/* 1-indexed mapping such as 1->2, 2->3, ... */
string permutationToCSV(const vector<int>& p) {
    stringstream ss;
    for (size_t i = 0; i < p.size(); ++i) {
        if (i) ss << ", ";
        ss << (i + 1) << "->" << (p[i] + 1);
    }
    return ss.str();
}

/* Exactly the 10 vertices that were selected for permutation. */
string selectedNodesToCSV(const vector<int>& p) {
    stringstream ss;
    bool first = true;

    for (int i = 0; i < (int)p.size(); ++i) {
        if (p[i] != i) {
            if (!first) ss << ", ";
            ss << i + 1;
            first = false;
        }
    }
    return ss.str();
}

int main(int argc, char** argv) {
    if (argc < 3) {
        cerr << "Usage: ./program input.txt output.csv\n";
        return 1;
    }

    const string inputFile = argv[1];
    const string outputFile = argv[2];

    const int TARGET_SUCCESSES = 10;
    const int MAX_MINUTES_PER_GRAPH = 30;
    const int MAX_TRIALS_PER_GRAPH = 1000000;

    // Time-based seed so repeated notebook runs do not reuse the same trials.
    random_device rd;
    mt19937 rng(rd());

    vector<GraphData> graphs = readGraphs(inputFile);

    cout << "Graphs loaded: " << graphs.size() << "\n";

    ofstream fout(outputFile);
    if (!fout) {
        cerr << "ERROR: Cannot create output CSV.\n";
        return 1;
    }

    /*
    ONE ROW = ONE SUCCESSFUL L1 + PERMUTATION + UNION RESULT.
    */
    fout
        << "sample_id,graph_id,n,success_number,trial_number,"
        << "selected_nodes,vertex_permutation,"
        << "l1_edges,permuted_l1_edges,union_edges,"
        << "chromatic_l1,chromatic_permuted_l1,chromatic_union,"
        << "l1_matrix,permuted_l1_matrix,union_matrix\n";

    long long sampleId = 0;

    for (size_t gi = 0; gi < graphs.size(); ++gi) {
        auto& G = graphs[gi];

        cout << "\n========================================\n";
        cout << "Graph " << gi + 1 << "/" << graphs.size()
             << ": " << G.id << "\n";
        cout << "Nodes: " << G.n << "\n";
        cout << "========================================\n";

        if (G.n < 10) {
            cout << "Skipping: fewer than 10 nodes.\n";
            continue;
        }

        auto start = Clock::now();
        auto deadline = start + chrono::minutes(MAX_MINUTES_PER_GRAPH);

        int chiOriginal = chromaticNumber(G.adj, deadline);

        if (chiOriginal == -1) {
            cout << "TIMEOUT on original graph.\n";
            continue;
        }

        cout << "L1 χ = " << chiOriginal << "\n";

        int successes = 0;
        int trials = 0;

        while (successes < TARGET_SUCCESSES &&
               trials < MAX_TRIALS_PER_GRAPH &&
               Clock::now() < deadline) {

            ++trials;

            // NEW random node selection + NEW permutation EVERY trial.
            vector<int> p = createPermutation(G.n, rng);
            vector<vector<int>> P = permuteGraph(G.adj, p);

            int chiP = chromaticNumber(P, deadline);
            if (chiP == -1) {
                cout << "TIMEOUT on permuted graph.\n";
                break;
            }

            vector<vector<int>> U = unionGraph(G.adj, P);
            int chiU = chromaticNumber(U, deadline);

            if (chiU == -1) {
                cout << "TIMEOUT on union graph.\n";
                break;
            }

            // EXACT acceptance rule from the original notebook.
            if (chiU == 8 || chiU == 9) {
                ++successes;
                ++sampleId;

                fout
                    << sampleId << ","
                    << "\"" << G.id << "\","
                    << G.n << ","
                    << successes << ","
                    << trials << ","
                    << "\"" << selectedNodesToCSV(p) << "\","
                    << "\"" << permutationToCSV(p) << "\","
                    << "\"" << edgesToCSV(G.adj) << "\","
                    << "\"" << edgesToCSV(P) << "\","
                    << "\"" << edgesToCSV(U) << "\","
                    << chiOriginal << ","
                    << chiP << ","
                    << chiU << ","
                    << "\"" << matrixToCSV(G.adj) << "\","
                    << "\"" << matrixToCSV(P) << "\","
                    << "\"" << matrixToCSV(U) << "\"\n";

                fout.flush();

                cout << "SUCCESS " << successes << "/10"
                     << " | trial=" << trials
                     << " | χ(L1)=" << chiOriginal
                     << " | χ(P)=" << chiP
                     << " | χ(union)=" << chiU << "\n";
            }

            if (trials % 100 == 0 && successes < TARGET_SUCCESSES) {
                cout << "Trial " << trials
                     << " | successes=" << successes
                     << " | last union χ=" << chiU << "\n";
            }
        }

        auto elapsed =
            chrono::duration_cast<chrono::seconds>(
                Clock::now() - start).count();

        cout << "Finished " << G.id
             << " | successes=" << successes << "/10"
             << " | trials=" << trials
             << " | time=" << elapsed << " sec\n";
    }

    fout.close();

    cout << "\n========================================\n";
    cout << "STAGE 1 FINISHED\n";
    cout << "Output: " << outputFile << "\n";
    cout << "========================================\n";

    return 0;
}
'''

with open(CPP_FILE, 'w') as f:
    f.write(cpp_source)

print('Created:', CPP_FILE)


Created: planar_permutation_boost_final.cpp


In [4]:
# 4. Compile Stage 1 program
import subprocess, os

EXE_FILE = "planar_permutation_boost_final"

result = subprocess.run(
    ["g++", "-std=c++17", "-O3", "-DNDEBUG", CPP_FILE, "-o", EXE_FILE],
    capture_output=True, text=True
)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Compilation failed.")

print("Compilation successful.")


Compilation successful.


In [5]:
# 5. Run Stage 1
# Live progress: C++ stdout is streamed line-by-line to this notebook cell.
# No Stage-1 search logic is changed here.

import subprocess

print("========== STAGE 1 START ==========")
print("Progress: every 100 trials; every successful permutation is printed immediately.")
print("Acceptance rule: union chromatic number = 8 or 9.")
print()

process = subprocess.Popen(
    ["./" + EXE_FILE, INPUT_FILE, STAGE1_CSV],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in process.stdout:
    print(line, end="", flush=True)

return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f"Stage 1 execution failed with return code {return_code}.")

print("\n========== STAGE 1 COMPLETE ==========")
print("Stage 1 CSV created:", STAGE1_CSV)


Streaming output truncated to the last 5000 lines.
Trial 946200 | successes=1 | last union χ=6
Trial 946300 | successes=1 | last union χ=6
Trial 946400 | successes=1 | last union χ=5
Trial 946500 | successes=1 | last union χ=5
Trial 946600 | successes=1 | last union χ=6
Trial 946700 | successes=1 | last union χ=6
Trial 946800 | successes=1 | last union χ=6
Trial 946900 | successes=1 | last union χ=6
Trial 947000 | successes=1 | last union χ=6
Trial 947100 | successes=1 | last union χ=6
Trial 947200 | successes=1 | last union χ=5
Trial 947300 | successes=1 | last union χ=6
Trial 947400 | successes=1 | last union χ=6
Trial 947500 | successes=1 | last union χ=5
Trial 947600 | successes=1 | last union χ=6
Trial 947700 | successes=1 | last union χ=6
Trial 947800 | successes=1 | last union χ=5
Trial 947900 | successes=1 | last union χ=6
Trial 948000 | successes=1 | last union χ=6
Trial 948100 | successes=1 | last union χ=6
Trial 948200 | successes=1 | last union χ=6
Trial 948300 | successes=

In [6]:
# 6. Inspect Stage 1 output
import pandas as pd

df_stage1 = pd.read_csv(STAGE1_CSV)

print("Stage-1 rows:", len(df_stage1))
print("Unique original graph IDs:", df_stage1["graph_id"].nunique())
print("Union chromatic numbers:")
print(df_stage1["chromatic_union"].value_counts().sort_index())

display(df_stage1.head(3))


Stage-1 rows: 391
Unique original graph IDs: 40
Union chromatic numbers:
chromatic_union
8    391
Name: count, dtype: int64


,sample_id,graph_id,n,success_number,trial_number,selected_nodes,vertex_permutation,l1_edges,permuted_l1_edges,union_edges,chromatic_l1,chromatic_permuted_l1,chromatic_union,l1_matrix,permuted_l1_matrix,union_matrix
0,1,ID1-12.56.1:,12,1,567,"1, 2, 3, 4, 6, 7, 8, 9, 10, 12","1->4, 2->8, 3->6, 4->9, 5->5, 6->7, 7->10, 8->...","1-3, 1-4, 1-8, 2-3, 2-4, 2-6, 2-7, 2-9, 2-10, ...","1-2, 1-5, 1-6, 1-9, 1-12, 2-5, 2-8, 2-9, 2-12,...","1-2, 1-3, 1-4, 1-5, 1-6, 1-8, 1-9, 1-12, 2-3, ...",4,4,8,0 0 1 1 0 0 0 1 0 0 0 0|0 0 1 1 0 1 1 0 1 1 1 ...,0 1 0 0 1 1 0 0 1 0 0 1|1 0 0 0 1 0 0 1 1 0 0 ...,0 1 1 1 1 1 0 1 1 0 0 1|1 0 1 1 1 1 1 1 1 1 1 ...
1,2,ID1-12.56.1:,12,2,1208,"1, 2, 3, 4, 5, 6, 7, 10, 11, 12","1->6, 2->11, 3->10, 4->7, 5->1, 6->12, 7->2, 8...","1-3, 1-4, 1-8, 2-3, 2-4, 2-6, 2-7, 2-9, 2-10, ...","1-3, 1-5, 1-9, 2-7, 2-11, 2-12, 3-5, 3-7, 3-9,...","1-3, 1-4, 1-5, 1-8, 1-9, 2-3, 2-4, 2-6, 2-7, 2...",4,4,8,0 0 1 1 0 0 0 1 0 0 0 0|0 0 1 1 0 1 1 0 1 1 1 ...,0 0 1 0 1 0 0 0 1 0 0 0|0 0 0 0 0 0 1 0 0 0 1 ...,0 0 1 1 1 0 0 1 1 0 0 0|0 0 1 1 0 1 1 0 1 1 1 ...
2,3,ID1-12.56.1:,12,3,2108,"1, 2, 3, 4, 5, 6, 7, 8, 9, 10","1->8, 2->5, 3->10, 4->6, 5->7, 6->9, 7->3, 8->...","1-3, 1-4, 1-8, 2-3, 2-4, 2-6, 2-7, 2-9, 2-10, ...","1-6, 1-8, 1-9, 1-10, 1-11, 2-4, 2-5, 2-6, 2-7,...","1-3, 1-4, 1-6, 1-8, 1-9, 1-10, 1-11, 2-3, 2-4,...",4,4,8,0 0 1 1 0 0 0 1 0 0 0 0|0 0 1 1 0 1 1 0 1 1 1 ...,0 0 0 0 0 1 0 1 1 1 1 0|0 0 0 1 1 1 1 0 0 0 0 ...,0 0 1 1 0 1 0 1 1 1 1 0|0 0 1 1 1 1 1 0 1 1 1 ...


## Stage 2 — Balanced augmentation

For every base successful sample:

1. Keep its L1 structure.
2. Generate a new random permutation of **all vertex labels**.
3. Apply that exact same relabeling to:
   - L1
   - the successful permuted L1
   - the union
4. Rebuild all three adjacency matrices.
5. Keep the three chromatic numbers unchanged.
6. Keep the original `selected_nodes` information as metadata, and also store the new augmentation relabeling.

This is **label augmentation**, not a new graph-generation step.


In [7]:
# 7. Balanced augmentation to approximately 5000 samples
import pandas as pd
import numpy as np
import random
from collections import defaultdict

TARGET_AUGMENTED = 5000
RANDOM_SEED = 20260916
rng = random.Random(RANDOM_SEED)

df = pd.read_csv(STAGE1_CSV)

if len(df) == 0:
    raise ValueError("Stage-1 CSV contains no successful samples.")

base_count = len(df)

# Balanced integer allocation:
# every base sample gets floor(TARGET/base_count) copies,
# and the first remainder samples get one extra copy.
base_repeats = TARGET_AUGMENTED // base_count
remainder = TARGET_AUGMENTED % base_count

repeat_counts = [base_repeats + (1 if i < remainder else 0)
                 for i in range(base_count)]

assert sum(repeat_counts) == TARGET_AUGMENTED

print("Base successful samples:", base_count)
print("Target augmented samples:", TARGET_AUGMENTED)
print("Minimum augmentations per base sample:", min(repeat_counts))
print("Maximum augmentations per base sample:", max(repeat_counts))

# -----------------------------
# Matrix / edge helper functions
# -----------------------------
def parse_matrix(s):
    rows = []
    for row in str(s).split("|"):
        rows.append([int(x) for x in row.strip().split()])
    return np.array(rows, dtype=np.int8)

def matrix_to_edges(M):
    n = M.shape[0]
    edges = []
    for i in range(n):
        for j in range(i + 1, n):
            if M[i, j] == 1:
                edges.append(f"{i+1}-{j+1}")
    return ", ".join(edges)

def permute_matrix(M, mapping):
    # mapping[old_label-1] = new_label-1
    n = M.shape[0]
    P = np.zeros((n, n), dtype=np.int8)

    for i in range(n):
        for j in range(n):
            if M[i, j]:
                ni = mapping[i]
                nj = mapping[j]
                P[ni, nj] = 1

    return P

def matrix_to_string(M):
    return "|".join(
        " ".join(str(int(x)) for x in row)
        for row in M
    )

def mapping_to_string(mapping):
    return ", ".join(
        f"{i+1}->{mapping[i]+1}"
        for i in range(len(mapping))
    )

def parse_selected_nodes(s):
    if pd.isna(s) or str(s).strip() == "":
        return []
    return [int(x.strip()) for x in str(s).split(",")]

# -----------------------------
# Generate augmented rows
# -----------------------------
augmented_rows = []
aug_sample_id = 0

for base_idx, row in df.iterrows():
    n = int(row["n"])

    L1 = parse_matrix(row["l1_matrix"])
    P1 = parse_matrix(row["permuted_l1_matrix"])

    if L1.shape != (n, n) or P1.shape != (n, n):
        raise ValueError(
            f"Matrix shape mismatch in base sample {row['sample_id']}: "
            f"expected {(n,n)}, got {L1.shape} and {P1.shape}"
        )

    # Same number of augmented samples for each base sample, as balanced as possible.
    for aug_no in range(repeat_counts[base_idx]):
        # Random bijection of ALL labels 1..n.
        mapping = list(range(n))
        rng.shuffle(mapping)

        aug_L1 = permute_matrix(L1, mapping)
        aug_P1 = permute_matrix(P1, mapping)
        aug_U = np.maximum(aug_L1, aug_P1).astype(np.int8)

        # Chromatic numbers are invariant under vertex relabeling,
        # so retain the original exact values.
        aug_sample_id += 1

        augmented_rows.append({
            "sample_id": aug_sample_id,
            "base_sample_id": int(row["sample_id"]),
            "original_graph_id": row["graph_id"],
            "n": n,
            "augmentation_number": aug_no + 1,
            "selected_nodes": row["selected_nodes"],
            "original_vertex_permutation": row["vertex_permutation"],
            "augmentation_vertex_relabeling": mapping_to_string(mapping),

            "l1_edges": matrix_to_edges(aug_L1),
            "permuted_l1_edges": matrix_to_edges(aug_P1),
            "union_of_l1_and_permutation": matrix_to_edges(aug_U),

            "chromatic_l1": int(row["chromatic_l1"]),
            "chromatic_permuted_l1": int(row["chromatic_permuted_l1"]),
            "chromatic_union": int(row["chromatic_union"]),

            "l1_matrix": matrix_to_string(aug_L1),
            "permuted_l1_matrix": matrix_to_string(aug_P1),
            "union_matrix": matrix_to_string(aug_U)
        })

df_aug = pd.DataFrame(augmented_rows)

assert len(df_aug) == TARGET_AUGMENTED

# Verify balanced counts.
counts = df_aug["base_sample_id"].value_counts()
print("Augmented rows:", len(df_aug))
print("Per-base minimum:", counts.min())
print("Per-base maximum:", counts.max())
print("Per-base distribution:")
print(counts.value_counts().sort_index())

df_aug.to_csv(AUGMENTED_CSV, index=False)

print("\nCreated:", AUGMENTED_CSV)
display(df_aug.head(3))


Base successful samples: 391
Target augmented samples: 5000
Minimum augmentations per base sample: 12
Maximum augmentations per base sample: 13
Augmented rows: 5000
Per-base minimum: 12
Per-base maximum: 13
Per-base distribution:
count
12     83
13    308
Name: count, dtype: int64

Created: augmented_5000_planar_permutation_results.csv


,sample_id,base_sample_id,original_graph_id,n,augmentation_number,selected_nodes,original_vertex_permutation,augmentation_vertex_relabeling,l1_edges,permuted_l1_edges,union_of_l1_and_permutation,chromatic_l1,chromatic_permuted_l1,chromatic_union,l1_matrix,permuted_l1_matrix,union_matrix
0,1,1,ID1-12.56.1:,12,1,"1, 2, 3, 4, 6, 7, 8, 9, 10, 12","1->4, 2->8, 3->6, 4->9, 5->5, 6->7, 7->10, 8->...","1->6, 2->12, 3->10, 4->4, 5->8, 6->5, 7->9, 8-...","1-2, 1-10, 1-12, 2-4, 2-5, 2-6, 2-10, 3-4, 3-7...","1-2, 1-5, 1-10, 2-3, 2-5, 2-7, 2-9, 2-11, 2-12...","1-2, 1-5, 1-10, 1-12, 2-3, 2-4, 2-5, 2-6, 2-7,...",4,4,8,0 1 0 0 0 0 0 0 0 1 0 1|1 0 0 1 1 1 0 0 0 1 0 ...,0 1 0 0 1 0 0 0 0 1 0 0|1 0 1 0 1 0 1 0 1 0 1 ...,0 1 0 0 1 0 0 0 0 1 0 1|1 0 1 1 1 1 1 0 1 1 1 ...
1,2,1,ID1-12.56.1:,12,2,"1, 2, 3, 4, 6, 7, 8, 9, 10, 12","1->4, 2->8, 3->6, 4->9, 5->5, 6->7, 7->10, 8->...","1->11, 2->2, 3->1, 4->9, 5->6, 6->12, 7->10, 8...","1-2, 1-3, 1-4, 1-7, 1-8, 1-9, 1-11, 2-3, 2-5, ...","1-3, 1-5, 1-9, 1-10, 1-12, 2-4, 2-5, 2-6, 2-8,...","1-2, 1-3, 1-4, 1-5, 1-7, 1-8, 1-9, 1-10, 1-11,...",4,4,8,0 1 1 1 0 0 1 1 1 0 1 0|1 0 1 0 1 0 1 0 1 1 0 ...,0 0 1 0 1 0 0 0 1 1 0 1|0 0 0 1 1 1 0 1 0 0 1 ...,0 1 1 1 1 0 1 1 1 1 1 1|1 0 1 1 1 1 1 1 1 1 1 ...
2,3,1,ID1-12.56.1:,12,3,"1, 2, 3, 4, 6, 7, 8, 9, 10, 12","1->4, 2->8, 3->6, 4->9, 5->5, 6->7, 7->10, 8->...","1->11, 2->4, 3->7, 4->9, 5->6, 6->12, 7->2, 8-...","1-5, 1-6, 1-7, 1-8, 1-9, 2-4, 2-9, 2-12, 3-4, ...","1-4, 1-6, 1-10, 1-11, 1-12, 2-5, 2-7, 2-8, 2-1...","1-4, 1-5, 1-6, 1-7, 1-8, 1-9, 1-10, 1-11, 1-12...",4,4,8,0 0 0 0 1 1 1 1 1 0 0 0|0 0 0 1 0 0 0 0 1 0 0 ...,0 0 0 1 0 1 0 0 0 1 1 1|0 0 0 0 1 0 1 1 0 1 0 ...,0 0 0 1 1 1 1 1 1 1 1 1|0 0 0 1 1 0 1 1 1 1 0 ...


In [8]:
# 8. Final validation
print("========== FINAL VALIDATION ==========")
print("Stage 1 samples:", len(df_stage1))
print("Stage 1 unique graph IDs:", df_stage1["graph_id"].nunique())
print("Stage 1 accepted union χ values:",
      sorted(df_stage1["chromatic_union"].unique().tolist()))

print("\nAugmented samples:", len(df_aug))
print("Augmented unique base samples:",
      df_aug["base_sample_id"].nunique())

# Check that augmented chromatic numbers stayed the same for every base sample.
base_chi = df.set_index("sample_id")[
    ["chromatic_l1", "chromatic_permuted_l1", "chromatic_union"]
]

for _, r in df_aug.iterrows():
    b = base_chi.loc[r["base_sample_id"]]
    assert int(r["chromatic_l1"]) == int(b["chromatic_l1"])
    assert int(r["chromatic_permuted_l1"]) == int(b["chromatic_permuted_l1"])
    assert int(r["chromatic_union"]) == int(b["chromatic_union"])

print("Chromatic-number invariance check: PASSED")
print("Balanced augmentation check: PASSED")
print("======================================")


========== FINAL VALIDATION ==========
Stage 1 samples: 391
Stage 1 unique graph IDs: 40
Stage 1 accepted union χ values: [8]

Augmented samples: 5000
Augmented unique base samples: 391
Chromatic-number invariance check: PASSED
Balanced augmentation check: PASSED


In [9]:
# 9. Download both CSV outputs
from google.colab import files

print("Downloading Stage 1 CSV...")
files.download(STAGE1_CSV)

print("Downloading augmented CSV...")
files.download(AUGMENTED_CSV)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>